# Figures for "Efficient strategies for accurate kinetics from biased simulations"

Every manuscript and SI figure, drawn from the tables in `data/` only. Those tables are
written by `scripts/export_paper_figure_data.py` from the committed analysis; `data/manifest.json`
records the commit and the sources of each table.

Run top to bottom (`uv run --group figures jupyter nbconvert --to notebook --execute --inplace
paper_figures/make_paper_figures.ipynb`). Output: vector PDFs in `pdf/`, 300 dpi PNG previews in `png/`.

Format: J. Chem. Phys. widths (one column 3.37 in, two columns 6.69 in), text at 8 pt or
larger, lines at 0.5 pt or wider. Units: errors are $\varepsilon = \ln k_{\rm est} - \ln k_0$
in natural-log units; costs are total simulated time in $\mu$s; free energies in $k_{\rm B}T$.

In [1]:
import json
import logging
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, NullFormatter

logging.getLogger("fontTools").setLevel(logging.ERROR)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
HERE = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd() / "paper_figures"
DATA, PDF, PNG = HERE / "data", HERE / "pdf", HERE / "png"
PDF.mkdir(exist_ok=True); PNG.mkdir(exist_ok=True)
MANIFEST = json.loads((DATA / "manifest.json").read_text())
print("data exported at commit", MANIFEST["commit"])

W1, W2 = 3.37, 6.69  # JCP one- and two-column widths (in)
WSI = 6.5            # text width of the supplementary document, so SI figures print unscaled
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Helvetica", "DejaVu Sans"], "font.size": 9,
    "axes.labelsize": 9, "axes.titlesize": 9, "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
    "mathtext.fontset": "custom", "mathtext.rm": "Helvetica", "mathtext.it": "Helvetica:italic", "mathtext.bf": "Helvetica:bold",
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.minor.width": 0.5, "ytick.minor.width": 0.5,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.minor.size": 1.5, "ytick.minor.size": 1.5,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "lines.linewidth": 1.0, "lines.markersize": 3.5, "legend.frameon": False, "legend.handlelength": 1.6,
    "legend.borderaxespad": 0.3, "legend.labelspacing": 0.25, "legend.handletextpad": 0.4,
    "pdf.fonttype": 42, "savefig.dpi": 300, "figure.dpi": 150,
    "figure.constrained_layout.h_pad": 0.02, "figure.constrained_layout.w_pad": 0.02,
})

CVS = ["Q", "RMSD", "Rg", "Ree"]
CVNAME = {"Q": "$Q$", "RMSD": "RMSD", "Rg": "$R_g$", "Ree": "$R_{ee}$"}
CVUNIT = {"Q": "$Q$", "RMSD": "RMSD (nm)", "Rg": "$R_g$ (nm)", "Ree": "$R_{ee}$ (nm)"}
CVCOL = {"Q": "#332288", "RMSD": "#117733", "Rg": "#DDAA33", "Ree": "#AA3377"}  # Tol palette, CV quality
EATR, G1, GREY, INK = "#0072B2", "#D55E00", "#8a8a8a", "#222222"  # gamma-fitting / gamma = 1 families
SEQ = plt.cm.YlGnBu
EPS = r"$\varepsilon = \ln k_{\rm est} - \ln k_0$"
COST = r"total simulated time ($\mu$s)"

ref = pd.read_csv(DATA / "reference.csv").iloc[0]
cvsum = pd.read_csv(DATA / "cv_summary.csv").set_index("cv")


def label(ax, s):
    '''Panel letter (a), (b), ... as a left-aligned title, so the layout reserves its space.

    Args:
        ax (matplotlib.axes.Axes): panel.
        s (str): letter without parentheses.
    '''
    ax.set_title(f"({s})", loc="left", fontweight="bold", pad=2)


def num(x, fmt="+.2f"):
    '''Format a number with a typographic minus sign.

    Args:
        x (float): value.
        fmt (str): format specification.
    '''
    return format(x, fmt).replace("-", "\u2212")


def accuracy_band(ax):
    '''Shade one natural-log unit around the reference and draw the zero line.

    Args:
        ax (matplotlib.axes.Axes): panel.
    '''
    ax.axhspan(-1, 1, color=GREY, alpha=0.15, lw=0, zorder=0)
    ax.axhline(0, color=INK, lw=0.5, zorder=0)


def logx(ax, lo, hi):
    '''Log x axis with decade ticks only.

    Args:
        ax (matplotlib.axes.Axes): panel.
        lo (float): lower limit.
        hi (float): upper limit.
    '''
    ax.set_xscale("log"); ax.set_xlim(lo, hi)
    ax.xaxis.set_major_locator(LogLocator(base=10, numticks=12))
    ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10), numticks=12))
    ax.xaxis.set_minor_formatter(NullFormatter())


def logy(ax, lo, hi):
    '''Log y axis with decade ticks only.

    Args:
        ax (matplotlib.axes.Axes): panel.
        lo (float): lower limit.
        hi (float): upper limit.
    '''
    ax.set_yscale("log"); ax.set_ylim(lo, hi)
    ax.yaxis.set_major_locator(LogLocator(base=10, numticks=12))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10), numticks=12))
    ax.yaxis.set_minor_formatter(NullFormatter())


def save(fig, name):
    '''Write the vector PDF and a PNG preview; report the size against the JCP width.

    The PDF is also written into the manuscript's figures directory when that directory is
    present, so that the figure the paper includes is always the one this notebook drew.

    Args:
        fig (matplotlib.figure.Figure): figure.
        name (str): file stem.
    '''
    fig.savefig(PDF / f"{name}.pdf")
    fig.savefig(PNG / f"{name}.png")
    manuscript = HERE.parent / "overleaf_best_rates_paper/figures"
    if manuscript.is_dir():
        fig.savefig(manuscript / f"{name}.pdf")
    w, h = fig.get_size_inches()
    print(f"{name}: {w:.2f} x {h:.2f} in")
    plt.close(fig)

data exported at commit 35d453c


## Figure 1. System, coordinates and why $\gamma < 1$

(a) Snapshots of the C$\alpha$ Gō model at a ladder of $Q$, each the dumped frame of one
unbiased first-passage run nearest that $Q$ (renders from `snapshots/q_ladder/`, table
`q_ladder_frames.csv`); blue N-terminal hairpin, vermilion helix, green C-terminal hairpin.
(b) Unbiased PMFs along the four biased CVs; grey fill, fraction of frames at each value
that are unfolded ($Q < 0.35$); dashed line, apparent barrier. (c) OPES-flooding at pace
1000: biased landscape $F + \langle V\rangle$ for the five BARRIER values on $Q$ and $R_{ee}$.
(d) Observed escape rate of every OPES set at pace 1000 against its acceleration factor;
on the diagonal the naive estimate is exact.

In [2]:
pmf = pd.read_csv(DATA / "pmf.csv")
ov = pd.read_csv(DATA / "overlays.csv")
ovE = pd.read_csv(DATA / "overlay_errors.csv")
osets = pd.read_csv(DATA / "opes_sets.csv")
hsets = pd.read_csv(DATA / "highbarrier_sets.csv")  # pace-1000 ladders run past the barrier, for panel (d) only
XLIM = {"Q": (0, 1), "RMSD": (0, 2.6), "Rg": (0.98, 2.4), "Ree": (1.5, 8)}

snap = pd.read_csv(DATA / "q_ladder_frames.csv")
from PIL import Image

def framed(path, box=1000):
    """Fit one cropped render into a square white canvas so every snapshot panel has the same frame."""
    im = Image.open(path).convert("RGB")
    im.thumbnail((box, box))
    canvas = Image.new("RGB", (box, box), "white")
    canvas.paste(im, ((box - im.width) // 2, (box - im.height) // 2))
    return canvas

fig = plt.figure(figsize=(W2, 3.85), layout="constrained")
gs = fig.add_gridspec(3, 1, height_ratios=(0.72, 1, 1.2))
row = gs[0].subgridspec(1, len(snap), wspace=0.02)
for j, r in enumerate(snap.itertuples()):
    ax = fig.add_subplot(row[j])
    ax.imshow(framed(HERE / "snapshots/q_ladder" / f"go_q{r.q}_run{r.run}_frame{r.frame}.png"))
    ax.set_axis_off()
    ax.set_title(f"$Q = {r.q:.2f}$", fontsize=8, pad=2)
    if j == 0:
        snap_left = ax
top = gs[1].subgridspec(1, 4); bot = gs[2].subgridspec(1, 4, width_ratios=(1, 1, 0.14, 0.86))
for j, cv in enumerate(CVS):
    ax = fig.add_subplot(top[j])
    p = pmf[pmf.cv == cv]
    ax.fill_between(p.x, 0, p.p_unfolded, transform=ax.get_xaxis_transform(), color=GREY, alpha=0.3, lw=0)
    ax.plot(p.x, p.F_kT, color=INK, lw=1.2)
    s = cvsum.loc[cv]
    ax.axvline(s.barrier_pos, color=INK, lw=0.6, ls="--")
    ax.set_xlim(*XLIM[cv]); ax.set_ylim(0, 10); ax.set_yticks([0, 2, 4, 6, 8])
    ax.set_xlabel(CVUNIT[cv])
    ax.text(0.97 if cv != "Q" else 0.05, 0.95, rf"$\Delta F^\ddagger$ = {s.barrier_kT:.1f} $k_{{\rm B}}T$" + "\n" + rf"$\gamma$ = {s.gamma_opes_ladder:.2f}",
            transform=ax.transAxes, ha="right" if cv != "Q" else "left", va="top",
            bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
    if j == 0:
        ax.set_ylabel(r"$F$ ($k_{\rm B}T$)")
    if j == 0:
        top_left = ax
for j, cv in enumerate(("Q", "Ree")):
    ax = fig.add_subplot(bot[j])
    p = pmf[pmf.cv == cv]
    ax.plot(p.x, p.F_kT, color=INK, lw=1.2, zorder=3)
    for i, b in enumerate((5, 7, 9, 11, 13)):
        o = ov[(ov.family == "opes") & (ov.cv == cv) & (ov.set_value == b)]
        ax.plot(o.x, o.F_plus_V_kT, color=SEQ(0.3 + 0.65 * i / 4), lw=1.0, label=f"{b}")
    ax.axvline(cvsum.loc[cv].barrier_pos, color=INK, lw=0.6, ls="--")
    ax.set_xlim(*((0.3, 1) if cv == "Q" else (1.5, 6.5))); ax.set_ylim(0, 10); ax.set_yticks([0, 2, 4, 6, 8])
    ax.set_xlabel(CVUNIT[cv])
    e = ovE[(ovE.family == "opesf") & (ovE.cv == cv)].set_index("estimator")
    # three short lines: upper left of the Q panel, upper right of the Ree panel, clear of every curve
    ax.text(0.03 if cv == "Q" else 0.97, 0.96,
            rf"naive: $\varepsilon$ = {num(e.loc['OPESf'].err_ln)}" + "\n"
            + rf"EATR-f: $\varepsilon$ = {num(e.loc['EATRf'].err_ln)}" + "\n"
            + rf"$\gamma$ = {e.loc['EATRf'].gamma:.2f}",
            transform=ax.transAxes, ha="left" if cv == "Q" else "right", va="top",
            bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
    if j == 0:
        ax.set_ylabel(r"$F + \langle V\rangle$ ($k_{\rm B}T$)")
        overlay_left = ax
    else:  # the legend sits in the empty lower right of the Ree panel, below the unfolded branch of F
        ax.legend(loc="lower right", title=r"BARRIER (kJ/mol)", title_fontsize=8, fontsize=8, ncol=3,
                  columnspacing=0.8, handlelength=1.0, labelspacing=0.2, borderpad=0.2, borderaxespad=0.3)
ax = fig.add_subplot(bot[3])
lim = (1.5, 9.1)
ax.plot(lim, lim, color=INK, lw=0.6, ls="--")
ax.text(3.6, 4.25, r"$\gamma = 1$", rotation=45, transform_rotates_text=True, ha="center", va="center")
for cv in CVS:
    g = osets[(osets.cv == cv) & (osets.pace == 1000)].sort_values("ln_exp_beta_v")
    ax.plot(g.ln_exp_beta_v, g.lnk_obs_us - ref.ln_k, "o-", color=CVCOL[cv], ms=3.5, lw=0.8)
    x = hsets[(hsets.cv == cv) & (hsets.barrier > 13)].sort_values("ln_exp_beta_v")
    if len(x):  # levels past the barrier: open symbols, joined to the ladder, no rate estimate uses them
        ax.plot(np.r_[g.ln_exp_beta_v.iloc[-1], x.ln_exp_beta_v], np.r_[g.lnk_obs_us.iloc[-1], x.lnk_obs_us] - ref.ln_k,
                "o-", color=CVCOL[cv], mfc="white", ms=3.5, lw=0.8)
    # Q is labelled right of and above the last level of its extended ladder, Ree above its third point, RMSD and Rg
    # right of the last level of their extended ladders: all clear of the diagonal, the frame and each other
    full = pd.concat([g, x])
    k, off, ha, va = {"Q": (-1, (6, 5), "left", "bottom"), "Ree": (2, (0, 6), "center", "bottom"),
                      "RMSD": (-1, (5, 3), "left", "bottom"), "Rg": (-1, (4, 0), "left", "center")}[cv]
    ax.annotate(CVNAME[cv], (full.ln_exp_beta_v.iloc[k], full.lnk_obs_us.iloc[k] - ref.ln_k), xytext=off,
                textcoords="offset points", ha=ha, va=va, color=CVCOL[cv])
ax.set_xlim(lim); ax.set_ylim(0.6, 6.4)
ax.set_xlabel(r"$\ln\langle e^{\beta V}\rangle$"); ax.set_ylabel(r"$\ln (k_{\rm obs}/k_0)$")
# one label per group of panels, set in the margin outside the axes
fig.canvas.draw()
fig.set_layout_engine("none")
for letter, ax0 in (("a", snap_left), ("b", top_left), ("c", overlay_left), ("d", ax)):
    bb = ax0.get_position()
    fig.text(max(bb.x0 - 0.052, 0.002), min(bb.y1 + 0.012, 0.995), f"({letter})", fontweight="bold",
             ha="left", va="top")
save(fig, "fig_system_and_gamma")

fig_system_and_gamma: 6.69 x 3.85 in


## Figure 2. Same data, two estimators: error against cost, and the fitted $\gamma$

(a) Well-tempered metadynamics grid (639 conditions of 50 to 200 runs, bias factors 2–20), iMetaD
and EATR on identical data. (b) Sets of static bias levels, OPES-flooding (one point per pace, five
BARRIER sets of 50 runs) and VES cutoff flooding (one point per number of runs per cutoff),
$\gamma = 1$ estimate and EATR-flooding. Black: unbiased reference (400 runs).
(c) The fitted $\gamma$ against deposition pace: every WT-MetaD condition with bias factor 5–20
(points) and the median at each pace (line); EATR-flooding over the five OPES levels at each pace
(circles); EATR-flooding over the six VES cutoffs (dotted line, square at the right edge).


In [3]:
ac = pd.read_csv(DATA / "accuracy_cost.csv")
# One encoding across Figs. 2 and 4: the marker shape carries the protocol, and the colour
# and the fill both carry the estimator, so that no distinction rests on telling two hues
# apart. Filled and blue is a fitted gamma; open and vermilion assumes gamma = 1. Row (c)
# shows the fitted gamma itself, so everything there is blue.
fig = plt.figure(figsize=(W2, 5.0), layout="constrained")
sfs = fig.subfigures(2, 1, height_ratios=(2, 1.08))
axes = sfs[0].subplots(2, 4, sharex=True, sharey="row")
for j, cv in enumerate(CVS):
    ax = axes[0, j]
    g = ac[(ac.family == "wtmetad") & (ac.cv == cv)]
    for est, col, z in (("iMetaD", G1, 1), ("EATR", EATR, 2)):
        d = g[g.estimator == est]
        ax.scatter(d.cost_us, d.err_ln, s=5, c=col, alpha=0.6, lw=0, zorder=z, rasterized=True)
    # untempered metadynamics: the same growing bias with no tempering, b = inf
    gu = ac[(ac.family == "untempered") & (ac.cv == cv)]
    for est, col, filled in (("iMetaD", G1, False), ("EATR", EATR, True)):
        d = gu[gu.estimator == est]
        ax.scatter(d.cost_us, d.err_ln, s=15, marker="^", facecolors=col if filled else "none",
                   edgecolors=col, lw=0.8, zorder=4)
    # frequency-adaptive metadynamics: six thresholds at b = 10, h0 = 0.19 kT
    gf = ac[(ac.family == "fametad") & (ac.cv == cv)]
    for est, col, filled in (("iMetaD", G1, False), ("EATR", EATR, True)):
        d = gf[gf.estimator == est]
        ax.scatter(d.cost_us, d.err_ln, s=13, marker="D", facecolors=col if filled else "none",
                   edgecolors=col, lw=0.8, zorder=4)
    ax.set_title(CVNAME[cv])
    ax = axes[1, j]
    for fam, mk in (("opesf", "o"), ("vesf", "s")):
        g = ac[(ac.family == fam) & (ac.cv == cv)]
        for est, col, filled in (("OPESf", G1, False), ("EATRf", EATR, True)):
            d = g[g.estimator == est]
            ax.scatter(d.cost_us, d.err_ln, s=12, marker=mk, facecolors=col if filled else "none",
                       edgecolors=col, lw=0.8, zorder=2)
for ax in axes.ravel():
    accuracy_band(ax)
    # the reference is a bar symbol: its standard error (0.05) is below the resolution of this axis
    ax.plot(ref.cost_us, 0, ls="", marker="|", ms=7, mew=1.4, color=INK, zorder=3)
    logx(ax, 0.05, 600)
axes[0, 0].set_ylim(-10, 4); axes[1, 0].set_ylim(-3.2, 1.8)
axes[0, 0].set_yticks([-8, -6, -4, -2, 0, 2, 4]); axes[1, 0].set_yticks([-3, -2, -1, 0, 1])
axes[0, 0].set_ylabel(r"$\varepsilon$, WT-MetaD grid"); axes[1, 0].set_ylabel(r"$\varepsilon$, static bias levels")
# rows (a) and (b) share one axis, so the label is written once, under the middle of the row
sfs[0].supxlabel(COST)

# (c) the fitted gamma against deposition pace, for the same data sets as (a) and (b)
gr = ac[(ac.family == "wtmetad") & (ac.estimator == "EATR") & (ac.bf >= 5)]
op = ac[(ac.family == "opesf") & (ac.estimator == "EATRf")]
ve = ac[(ac.family == "vesf") & (ac.estimator == "EATRf")]
gax = sfs[1].subplots(1, 4, sharey=True)
rng = np.random.default_rng(0)
for j, cv in enumerate(CVS):
    ax = gax[j]
    g = gr[gr.cv == cv]
    ax.scatter(g.pace * np.exp(rng.uniform(-0.15, 0.15, len(g))), g.gamma, s=5, c=EATR,
               alpha=0.35, lw=0, rasterized=True, zorder=1)
    med = g.groupby("pace").gamma.median()
    ax.plot(med.index, med.values, ls="-", lw=1.2, color=INK, zorder=3)
    o = op[op.cv == cv].sort_values("pace")
    ax.plot(o.pace, o.gamma, ls="--", marker="o", ms=4, lw=0.9, color=EATR, zorder=3)
    # the VES bias is static, so its single gamma is a level drawn to the right edge
    v = ve[(ve.cv == cv) & (ve.n_label == 200)].gamma.iloc[0]
    ax.axhline(v, color=EATR, ls=":", lw=1.0, zorder=2)
    ax.plot([1.45e5], [v], ls="", marker="s", ms=3.5, color=EATR, clip_on=False, zorder=4)
    ax.axhline(1, color=INK, lw=0.5, zorder=0)
    logx(ax, 60, 1.7e5); ax.set_ylim(0, 1.06)
gax[0].set_ylabel(r"fitted $\gamma$")
sfs[1].supxlabel("deposition pace (steps)")
# the columns are named by their titles, so only the three rows are lettered
label(axes[0, 0], "a"); label(axes[1, 0], "b"); label(gax[0], "c")
handles = [Line2D([], [], ls="", marker=".", ms=6, color=INK, label="WT-MetaD grid"),
           Line2D([], [], ls="", marker="^", ms=4, mfc="none", mew=0.8, color=INK, label=r"untempered, $b = \infty$"),
           Line2D([], [], ls="", marker="D", ms=3.5, mfc="none", mew=0.8, color=INK, label="frequency-adaptive"),
           Line2D([], [], ls="", marker="o", ms=4, color=INK, label="OPES-flooding"),
           Line2D([], [], ls="", marker="s", ms=4, color=INK, label="VES-flooding"),
           Line2D([], [], ls="-", lw=1.2, color=INK, label="WT-MetaD grid, median per pace"),
           Line2D([], [], ls="", marker="o", ms=4, color=EATR, label="EATR, EATR-f (filled)"),
           Line2D([], [], ls="", marker="o", ms=4, mfc="white", mew=0.8, color=G1, label=r"iMetaD, $\gamma = 1$ (open)"),
           Line2D([], [], ls="", marker="|", ms=7, mew=1.4, color=INK, label="unbiased reference")]
fig.legend(handles=handles, loc="outside lower center", ncol=3, columnspacing=1.0, handletextpad=0.3)
save(fig, "fig_error_vs_cost")


fig_error_vs_cost: 6.69 x 5.00 in


## Figure 3. How much data each strategy needs, for $Q$ and $R_{ee}$

(a) Fraction of random subsets whose estimate lies within one ln unit of the reference, against
the median cost of a subset. (b) The smallest cost at which nine subsets in ten land within one
ln unit and stay there, with the number of runs beside each point; arrows mark the strategies
that never reach it. Subset sizes step by about a factor of 1.5, so each minimum is resolved to that
factor. WT-MetaD at two slow paces, OPES per BARRIER level, VES per cutoff; all four CVs
in the supplementary version below.


In [4]:
# Minimum data for the best and the worst CV. Encoding as in Fig. 2: shape carries the
# protocol (circle WT-MetaD, triangle OPES, square VES), colour and fill the estimator
# (filled blue fitted gamma, open vermilion gamma = 1). All four CVs: the supplementary version.
HL = ["Q", "Ree"]
STY = {"wtmetad": "o", "opes": "^", "ves": "s"}
EST = {"EATR": (EATR, True), "EATRf": (EATR, True), "iMetaD": (G1, False), "OPESf": (G1, False)}
ROWS = [("WT-MetaD, pace 30000", r"WT-MetaD $3\times10^4$", "wtmetad"),
        ("WT-MetaD, pace 100000", r"WT-MetaD $10^5$", "wtmetad"),
        ("OPES", "OPES-flooding", "opes"), ("VES", "VES-flooding", "ves")]
mp = pd.read_csv(DATA / "mindata_points.csv")
mp = mp[mp.estimator.isin(list(EST))]


def mindata_chart(ax, cv):
    """Draw the smallest cost at which nine subsets in ten land within one ln unit, per protocol.

    Args:
        ax (matplotlib.axes.Axes): panel.
        cv (str): CV key in `mp`.
    """
    for y, (prot, lab, key) in enumerate(ROWS):
        for est, (col, filled) in EST.items():
            r = mp[(mp.cv == cv) & (mp.protocol == prot) & (mp.estimator == est)]
            if r.empty:
                continue
            if bool(r.never.iloc[0]):        # criterion never met: park an arrow at the edge
                ax.plot(200, -y, marker=">", ms=5, color=col, mfc="none", mew=0.8, clip_on=False)
            else:
                x = float(r.min_cost_us.iloc[0])
                ax.plot(x, -y, ls="", marker=STY[key], ms=5, color=col, mfc=col if filled else "white", mew=0.8)
                # the two estimators can land at nearly the same cost, so their run counts
                # are annotated on opposite sides of the marker rather than on top of each other
                ax.annotate(f"{int(r.n_at_min.iloc[0])}", (x, -y), xytext=(0, 4 if filled else -11),
                            textcoords="offset points", ha="center", fontsize=8, color=col)
    ax.set_ylim(-len(ROWS) + 0.4, 0.6)
    ax.set_yticks([-y for y in range(len(ROWS))])
    ax.grid(axis="y", color=GREY, alpha=0.3, lw=0.4)
    logx(ax, 0.1, 300)
    ax.set_title(CVNAME[cv])


MD_HANDLES = [Line2D([], [], ls="", marker="o", ms=4.5, color=INK, label="WT-MetaD"),
              Line2D([], [], ls="", marker="^", ms=4.5, color=INK, label="OPES-flooding"),
              Line2D([], [], ls="", marker="s", ms=4.5, color=INK, label="VES-flooding"),
              Line2D([], [], ls="", marker=">", ms=4.5, mfc="none", mew=0.8, color=INK, label="never reached"),
              Line2D([], [], ls="", marker="o", ms=4.5, color=EATR, label="EATR, EATR-f (filled)"),
              Line2D([], [], ls="", marker="o", ms=4.5, mfc="white", mew=0.8, color=G1, label=r"iMetaD, $\gamma = 1$ (open)")]
md_ = pd.read_csv(DATA / "mindata_curves.csv")
LST = {"wtmetad": dict(ls="-", ms=3.0, lw=1.1), "opes": dict(ls="--", ms=3.5, lw=0.9), "ves": dict(ls=":", ms=3.5, lw=1.0)}


def fraction_chart(ax, cv):
    """Draw the fraction of random subsets within one ln unit against the median cost of a subset.

    Args:
        ax (matplotlib.axes.Axes): panel.
        cv (str): CV key in `md_`.
    """
    for prot, st_ in LST.items():
        for est, (col, filled) in EST.items():
            d = md_[(md_.protocol == prot) & (md_.cv == cv) & (md_.estimator == est)].sort_values("n")
            if d.empty:
                continue
            ax.plot(d.cost_us, d.p1, **st_, marker=STY[prot], color=col, mfc=col if filled else "white", mew=0.8)
    ax.axhline(0.9, color=INK, lw=0.5, ls=":")
    ax.set_ylim(0, 1.03)
    logx(ax, 0.1, 300)
    ax.set_title(CVNAME[cv])


MD_LINES = [Line2D([], [], color=INK, marker=STY[k], **LST[k], label=n) for k, n in
            (("wtmetad", "WT-MetaD"), ("opes", "OPES-flooding"), ("ves", "VES-flooding"))]
fig = plt.figure(figsize=(W1, 4.1), layout="constrained")
# the rows share one x axis, so its label is written once under the middle; the label
# belongs to a subfigure so that the figure's outside legend sits below it
sf = fig.subfigures(1, 1)
axes = sf.subplots(2, 2, sharex=True, sharey="row"); sf.supxlabel(COST)
for j, cv in enumerate(HL):
    fraction_chart(axes[0, j], cv)
    mindata_chart(axes[1, j], cv)
    axes[1, j].set_title(None)
axes[0, 0].set_ylabel("fraction within\n1 ln unit")
# the row labels are shared, so they are set once; a blank list on the twin would blank both
axes[1, 0].set_yticklabels([lab for _, lab, _ in ROWS])
label(axes[0, 0], "a"); label(axes[1, 0], "b")
fig.legend(handles=MD_LINES + MD_HANDLES[3:], loc="outside lower center", ncol=2, columnspacing=1.0, handletextpad=0.3, fontsize=8)
save(fig, "fig_minimum_data")


fig_minimum_data: 3.37 x 4.10 in


## Figure 4. Where a growing bias works

Every grid condition by how much barrier its bias left standing and by its deposition
interval in units of the relaxation time of the CV, coloured by whether EATR lands within
one ln unit; failures are triangles pointing the way the rate errs (up: $\ln k_{\rm est}^{\rm EATR} > \ln k_0 + 1$, down: $\ln k_{\rm est}^{\rm EATR} < \ln k_0 - 1$). The supplement's version adds the growth law's escape-time prediction with the
two prospective runs.


In [5]:
dc = pd.read_csv(DATA / "design_conditions.csv")
pro = pd.read_csv(DATA / "prospective.csv")
good = dc.eatr_err.abs() <= 1
# the sign of a failure separates its mechanism: too fast from deposition, too slow from overfilling
FAIL = [(dc.eatr_err > 1, "^", r"$\ln k_{\rm est}^{\rm EATR} > \ln k_0 + 1$"),
        (dc.eatr_err < -1, "v", r"$\ln k_{\rm est}^{\rm EATR} < \ln k_0 - 1$")]


def conditions_panel(ax):
    """Every grid condition by the barrier its bias left and its pace against the relaxation time.

    Args:
        ax (matplotlib.axes.Axes): panel.
    """
    ax.axhspan(2, 12, xmin=0, xmax=1, color=EATR, alpha=0.07, lw=0)
    ax.axvspan(0.3, 1e3, color=EATR, alpha=0.07, lw=0)
    d2 = dc[good]
    ax.scatter(d2.tau_over_relax, d2.residual_kT, s=10, c=EATR, alpha=0.55, lw=0, rasterized=True)
    for m, mk, _ in FAIL:
        d2 = dc[m]
        ax.scatter(d2.tau_over_relax, d2.residual_kT, s=24, marker=mk, c=G1, alpha=0.85, lw=0, rasterized=True)
    ax.axhline(2, color=INK, lw=0.6, ls="--"); ax.axvline(0.3, color=INK, lw=0.6, ls="--")
    logx(ax, 1e-3, 30); ax.set_ylim(-7.5, 7)
    ax.set_xlabel(r"$\tau$ / CV relaxation time"); ax.set_ylabel(r"barrier left, $\Delta F^\ddagger - \gamma V$ ($k_{\rm B}T$)")


# main text: the two conditions alone, one column
fig, ax = plt.subplots(figsize=(W1, 2.5), layout="constrained")
conditions_panel(ax)
ax.legend(handles=[Line2D([], [], ls="", marker="o", ms=3.5, color=EATR, label="EATR within 1 ln unit")]
          + [Line2D([], [], ls="", marker=mk, ms=5, color=G1, label=lab) for _, mk, lab in FAIL],
          loc="lower right", fontsize=8)
save(fig, "fig_design_rule")

# supplement: the growth law's escape times beside the conditions, with the prospective runs
fig, axes = plt.subplots(2, 1, figsize=(W1, 4.3), layout="constrained")
ax = axes[0]
lim = (5e-5, 1)
x = np.array(lim)
ax.fill_between(x, x / 2, x * 2, color=GREY, alpha=0.2, lw=0)
ax.plot(x, x, color=INK, lw=0.5)
for cv in CVS:
    g = dc[dc.cv == cv]
    ax.scatter(g.t_star_unbroadened_ps / 1e6, g.t_obs_mean_ps / 1e6, s=4, c=CVCOL[cv], alpha=0.7, lw=0, rasterized=True)
for _, r in pro.iterrows():
    ax.scatter(r.pred_t_star_ns / 1e3, r.obs_mean_escape_ns / 1e3, marker="*", s=70, c=CVCOL[r.cv], edgecolors=INK, lw=0.5, zorder=4)
logx(ax, *lim); logy(ax, *lim)
ax.set_xlabel(r"predicted $t^\ast$ ($\mu$s)"); ax.set_ylabel(r"observed escape time ($\mu$s)")
conditions_panel(axes[1])
for i, ax in enumerate(axes):
    label(ax, "ab"[i])
handles = [Line2D([], [], ls="", marker="o", ms=3.5, color=CVCOL[cv], label=CVNAME[cv]) for cv in CVS]
handles.append(Line2D([], [], ls="", marker="*", ms=8, color=GREY, mec=INK, mew=0.5, label="prospective"))
handles += [Line2D([], [], ls="", marker=mk, ms=5, color=G1, label=f"(b) {lab}") for _, mk, lab in FAIL]
fig.legend(handles=handles, loc="outside lower center", ncol=3, columnspacing=0.9,
           handletextpad=0.3, fontsize=8)
save(fig, "figS_design_rule")
sel = (dc.residual_kT >= 2) & (dc.tau_over_relax >= 0.3)
print(f"inside both conditions: {sel.sum()} conditions, within 1 ln unit "
      f"{(dc[sel].eatr_err.abs() <= 1).mean():.3f}; outside {(dc[~sel].eatr_err.abs() <= 1).mean():.3f}")


fig_design_rule: 3.37 x 2.50 in


figS_design_rule: 3.37 x 4.30 in
inside both conditions: 273 conditions, within 1 ln unit 0.993; outside 0.757


## Figure 5. The error–cost front, for $Q$ and $R_{ee}$

Pooled root-mean-square error of $\ln k$ over independent subsets against the median cost
of the production runs. WT-MetaD: the conditions the design rule recommends a priori; flooding: all paces.
Faint points: every grid condition and subset size (bias factors 5–20). Black line: the front.
All four CVs are in the supplementary version below.


In [6]:
# The error-cost front for the best and the worst CV, same encoding as Figs. 2 and 3.
st = pd.read_csv(DATA / "pareto_strategies.csv")
fr = pd.read_csv(DATA / "pareto_front.csv")
bg = pd.read_csv(DATA / "pareto_background.csv")
PROT = {"WT-MetaD (recommended set)": "wtmetad", "OPES": "opes", "VES": "ves"}


def front_chart(ax, cv):
    """Draw every strategy's RMSE against its production cost, with the front.

    Args:
        ax (matplotlib.axes.Axes): panel.
        cv (str): CV key in the pareto tables.
    """
    b = bg[bg.cv == cv]
    ax.scatter(b.cost_us, b.rmse, s=2, c=[EATR if e == "EATR" else G1 for e in b.estimator],
               alpha=0.12, lw=0, rasterized=True)
    for prot, key in PROT.items():
        for est, (col, filled) in EST.items():
            d = st[(st.cv == cv) & (st.protocol == prot) & (st.estimator == est)]
            ax.plot(d.cost_us, d.rmse, ls="", marker=STY[key], ms=4.5, color=col,
                    mfc=col if filled else "white", mew=0.8, zorder=3)
    f = fr[fr.cv == cv].sort_values("cost_us")
    ax.plot(f.cost_us, f.rmse, color=INK, lw=1.0, zorder=2)
    ax.axhline(1, color=INK, lw=0.5, ls=":"); ax.axhline(0.5, color=INK, lw=0.5, ls=":")
    logx(ax, 1e-3, 300); logy(ax, 0.05, 10)
    ax.set_xticks([1e-2, 1, 1e2])  # every other decade, so the labels do not collide
    ax.set_title(CVNAME[cv])


FR_HANDLES = [Line2D([], [], ls="", marker="o", ms=4.5, color=INK, label="WT-MetaD"),
              Line2D([], [], ls="", marker="^", ms=4.5, color=INK, label="OPES-flooding"),
              Line2D([], [], ls="", marker="s", ms=4.5, color=INK, label="VES-flooding"),
              Line2D([], [], ls="", marker="o", ms=4.5, color=EATR, label="EATR, EATR-f (filled)"),
              Line2D([], [], ls="", marker="o", ms=4.5, mfc="white", mew=0.8, color=G1, label=r"iMetaD, $\gamma = 1$ (open)"),
              Line2D([], [], color=INK, lw=1.0, label="front")]
COSTP = r"cost ($\mu$s)"
fig = plt.figure(figsize=(W1, 3.1), layout="constrained")
sf = fig.subfigures(1, 1)
axes = sf.subplots(1, 2, sharey=True); sf.supxlabel(COSTP)
for ax, cv in zip(axes, HL):
    front_chart(ax, cv)
axes[0].set_ylabel(r"RMSE of $\ln k$")
fig.legend(handles=FR_HANDLES, loc="outside lower center", ncol=2, columnspacing=1.0, handletextpad=0.3, fontsize=8)
save(fig, "fig_error_cost_front")


fig_error_cost_front: 3.37 x 3.10 in


## Figure S(temperature). Static bias levels across temperature

(a) Free energy along $Q$ from the well-tempered scan at 312, 300, 285 and 270 K, both seeds;
the triangle marks the top of the mean profile. (b) Rate against the barrier increase from
312 K with the parameter-free prediction: EATR-flooding and $\gamma = 1$ on the five OPES-flooding
levels (means of bootstrap resamples), and EATR on single well-tempered conditions. (c) The
flooding fit to all runs, without resampling, with three, four and five bias sets; dotted,
the five-set value.


In [7]:
tr = pd.read_csv(DATA / "temperature_rates.csv")
tpmf = pd.read_csv(DATA / "temperature_pmf.csv")
tconv = pd.read_csv(DATA / "temperature_convergence.csv")
lad = tr[tr.family == "opes ladder"]
TEMPS = [312, 300, 285, 270]
TCOL = {T: SEQ(0.4 + 0.6 * i / 3) for i, T in enumerate(TEMPS)}

fig, axes = plt.subplots(1, 3, figsize=(W2, 2.5), layout="constrained")
# (a) the free energy the barriers are read from, both seeds, with the top of each mean profile marked
ax = axes[0]
for T in TEMPS:
    g = tpmf[tpmf.temperature_K == T]
    for k, (seed, sg) in enumerate(g.groupby("seed")):
        ax.plot(sg.q, sg.F_kT, color=TCOL[T], lw=1.0, alpha=0.85, label=f"{T} K" if k == 0 else None)
    m = g.groupby("q").F_kT.mean()
    q, f = m.index.to_numpy(), m.to_numpy()
    fold = np.argmin(np.where(q > 0.65, f, np.inf)); unf = np.argmin(np.where(q < 0.35, f, np.inf))
    lo, hi = sorted((fold, unf)); top = lo + int(np.argmax(f[lo:hi + 1]))
    ax.plot(q[top], f[top], "v", ms=5, color=TCOL[T], mec="white", mew=0.6, zorder=5)
ax.set_xlim(0.15, 1.0); ax.set_ylim(-1, 21)
ax.set_xlabel("$Q$"); ax.set_ylabel(r"$F$ ($k_{\rm B}T$)")
ax.legend(loc="upper right", handlelength=1.2)
# (b) the rate follows the barrier with no free parameter when gamma is fitted
ax = axes[1]
x = np.array([-0.5, 11.0])
ax.plot(x, ref.ln_k - x, color=INK, lw=0.8, ls="--", label=r"$-\Delta$(barrier)", zorder=1)
for est, col, fc, mk, lab in (("gamma = 1", G1, "white", "s", r"$\gamma = 1$"),
                              ("EATR-flooding", EATR, EATR, "o", "EATR-flooding")):
    g = lad[lad.estimator == est]
    ax.errorbar(g.d_barrier_kT, g.ln_k, yerr=g.ln_k_sd, fmt=mk, color=col, mfc=fc, mew=0.8, ms=4,
                capsize=2, elinewidth=0.7, lw=0, label=lab, zorder=3)
for fam, mk, fc, lab in (("wtmetad single, outside rule", "^", "white", r"WT-MetaD, $<2\,k_{\rm B}T$ left"),
                         ("wtmetad single, inside rule", "v", EATR, r"WT-MetaD, $>2\,k_{\rm B}T$ left")):
    g = tr[(tr.family == fam) & (tr.estimator == "EATR")]
    ax.plot(g.d_barrier_kT, g.ln_k, mk, ms=4.5, color=EATR, mfc=fc, mew=0.8, label=lab, zorder=2)
g = tr[(tr.family == "wtmetad single, R_ee") & (tr.estimator == "EATR")]  # one WT-MetaD condition along R_ee per temperature
# an open diamond on top: at 300 K the R_ee and Q single fills coincide to 0.01 and a filled marker would hide one
ax.plot(g.d_barrier_kT, g.ln_k, "D", ms=6, color=CVCOL["Ree"], mfc="none", mew=1.1, label=r"WT-MetaD along $R_{ee}$", zorder=6)
ax.set_xlim(-0.5, 11.5); ax.set_ylim(bottom=-12.6); ax.set_yticks(np.arange(-12, 1, 2))  # room for the legend below the 285 K points
ax.set_xlabel(r"barrier increase from 312 K ($k_{\rm B}T$)")
ax.set_ylabel(r"$\ln (k / \mu{\rm s}^{-1})$")
ax.legend(loc="lower left", fontsize=7, handletextpad=0.3)
# (c) how many bias sets the fit needs: point fits to all runs, dotted at the five-set value
ax = axes[2]
for T in TEMPS:
    g = tconv[tconv.temperature_K == T]
    ax.plot(g.n_sets, g.ln_k, "o-", ms=4, lw=1.0, color=TCOL[T])
    ax.axhline(g.ln_k_all_sets.iloc[0], color=TCOL[T], lw=0.7, ls=":")
ax.set_xticks([3, 4, 5])
ax.set_xlabel("bias sets used in the fit")
ax.set_ylabel(r"$\ln (k / \mu{\rm s}^{-1})$")
for i, ax in enumerate(axes):
    label(ax, "abc"[i])
save(fig, "figS_temperature")


figS_temperature: 6.69 x 2.50 in


## Figure S0. The range of the fitted $\gamma$ (supplementary)

Median and 10th–90th percentile of $\gamma$ for each CV from WT-MetaD at slow and fast paces,
the OPES paces and the VES fits at 10–200 runs per cutoff; dashed,
$\gamma = 0.8$. The pace dependence of the same fits is Fig. 2c.


In [8]:
ac = pd.read_csv(DATA / "accuracy_cost.csv")
# gamma is a fitted quantity in every series here, so the colour carries the CV and the
# marker the protocol; the pace dependence of these fits is Fig. 2c.
gr = ac[(ac.family == "wtmetad") & (ac.estimator == "EATR") & (ac.bf >= 5)]
op = ac[(ac.family == "opesf") & (ac.estimator == "EATRf")]
ve = ac[(ac.family == "vesf") & (ac.estimator == "EATRf")]
# the range of gamma each protocol returns for each CV: median and 10th-90th percentile
fig, ax = plt.subplots(figsize=(W1, 3.4), layout="constrained")
SERIES = [("WT-MetaD, pace $\\geq 10^4$", "o", True, lambda cv: gr[(gr.cv == cv) & (gr.pace >= 1e4)].gamma),
          ("WT-MetaD, pace $\\leq 10^3$", "o", False, lambda cv: gr[(gr.cv == cv) & (gr.pace <= 1e3)].gamma),
          ("OPES-flooding, eight paces", "^", True, lambda cv: op[op.cv == cv].gamma),
          ("VES-flooding, 10–200 runs/cutoff", "s", True, lambda cv: ve[ve.cv == cv].gamma)]
OFF = np.linspace(-0.3, 0.3, len(SERIES))
for j, cv in enumerate(CVS):
    for k, (name, mk, filled, pick) in enumerate(SERIES):
        y = pick(cv).to_numpy()
        lo, mid, hi = np.percentile(y, [10, 50, 90])
        x = j + OFF[k]
        ax.plot([x, x], [lo, hi], color=CVCOL[cv], lw=1.0, zorder=2)
        ax.plot([x], [mid], ls="", marker=mk, ms=4.5, color=CVCOL[cv],
                mfc=CVCOL[cv] if filled else "white", mew=0.8, zorder=3)
ax.axhline(1, color=INK, lw=0.5, zorder=0)
ax.axhline(0.8, color=INK, lw=0.6, ls="--", zorder=0)
ax.set_xticks(range(4)); ax.set_xticklabels([CVNAME[cv] for cv in CVS]); ax.set_xlim(-0.5, 3.5)
ax.set_ylim(0, 1.06); ax.set_ylabel(r"fitted $\gamma$")
handles = [Line2D([], [], ls="", marker=mk, ms=4.5, color=INK,
                  mfc=INK if filled else "white", mew=0.8, label=name) for name, mk, filled, _ in SERIES]
fig.legend(handles=handles, loc="outside lower center", ncol=1, fontsize=8, handletextpad=0.3)
save(fig, "figS_gamma_range")
for cv in CVS:
    print(cv, " ".join(f"{name.split(',')[0]}: {np.median(pick(cv)):.2f}" for name, _, _, pick in SERIES))


figS_gamma_range: 3.37 x 3.40 in
Q WT-MetaD: 0.90 WT-MetaD: 0.73 OPES-flooding: 0.89 VES-flooding: 0.81
RMSD WT-MetaD: 0.85 WT-MetaD: 0.56 OPES-flooding: 0.83 VES-flooding: 0.80
Rg WT-MetaD: 0.74 WT-MetaD: 0.59 OPES-flooding: 0.68 VES-flooding: 0.64
Ree WT-MetaD: 0.35 WT-MetaD: 0.38 OPES-flooding: 0.37 VES-flooding: 0.71


## Figure S0b. How much data each strategy needs, all four CVs

Top: fraction of random subsets whose estimate lies within one ln unit of the reference,
against the median cost of a subset. Bottom: the smallest cost at which nine subsets in
ten land within one ln unit and stay there, with the number of runs beside each point;
arrows mark the strategies that never reach it.
WT-MetaD: mean over the 160 grid conditions per CV (bias factors 2–20) at each number of runs;
OPES: mean over paces at each number of runs per BARRIER; VES: runs per cutoff.

In [9]:
# Same encoding as Fig. 3, whose chart functions draw both rows for all four CVs.
fig = plt.figure(figsize=(WSI, 3.6), layout="constrained")
# one x label under the middle of the bottom row, on a subfigure so the legend sits below it
sf = fig.subfigures(1, 1)
axes = sf.subplots(2, 4, sharex=True); sf.supxlabel(COST)
for j, cv in enumerate(CVS):
    fraction_chart(axes[0, j], cv)
    mindata_chart(axes[1, j], cv)
    axes[1, j].set_title(None)
axes[0, 0].set_ylabel("fraction within\n1 ln unit")
axes[1, 0].set_yticklabels([lab for _, lab, _ in ROWS])
for ax in axes[1, 1:]:
    ax.tick_params(labelleft=False)
# the columns are named by their titles, so only the two rows are lettered
label(axes[0, 0], "a"); label(axes[1, 0], "b")
fig.legend(handles=MD_LINES + MD_HANDLES[3:], loc="outside lower center", ncol=3, columnspacing=1.0)
save(fig, "figS_minimum_data")


figS_minimum_data: 6.50 x 3.60 in


## Figure S0c. The error–cost front, all four CVs

Pooled root-mean-square error of $\ln k$ over independent subsets against the median cost
of the production runs. WT-MetaD: the conditions the design rule recommends a priori; flooding: all paces.
Faint points: every grid condition and subset size (bias factors 5–20). Black line: the front.

In [10]:
# The front for every CV, drawn by the Fig. 5 chart function.
fig, axes = plt.subplots(1, 4, figsize=(WSI, 2.4), sharey=True, layout="constrained")
for j, cv in enumerate(CVS):
    front_chart(axes[j], cv)
    axes[j].set_xlabel(COSTP)
    label(axes[j], "abcd"[j])
axes[0].set_ylabel(r"RMSE of $\ln k$")
fig.legend(handles=FR_HANDLES, loc="outside lower center", ncol=4, columnspacing=0.8, handletextpad=0.3)
save(fig, "figS_error_cost_front")


figS_error_cost_front: 6.50 x 2.40 in


## Figure S1. Biased landscapes for every protocol

Unbiased PMF (black) and $F + \langle V\rangle$ per set (light to dark: weaker to stronger bias).
Top: WT-MetaD ($b = 10$, $h_0 = 0.5$ kJ/mol) at three paces, bias averaged near escape. Middle:
OPES-flooding at pace 1000. Bottom: VES cutoff flooding. Dashed: apparent barrier.

In [11]:
XL = {"Q": (0.3, 1), "Ree": (1.5, 5.5), "Rg": (0.98, 1.8)}
rows = [("imetad", (100, 1000, 100000), "pace {} steps", ("Q", "Ree")),
        ("opes", (5, 7, 9, 11, 13), "BARRIER {} kJ/mol", ("Q", "Ree", "Rg")),
        ("ves", (5, 7, 9, 11, 13, 15), "$F_c$ = {} kJ/mol", ("Q", "Ree", "Rg"))]
fig, axes = plt.subplots(3, 3, figsize=(WSI, 5.8), layout="constrained")
k = 0
for i, (fam, vals, fmt, cvs) in enumerate(rows):
    for j in range(3):
        ax = axes[i, j]
        if j >= len(cvs):
            ax.axis("off")
            ax.legend(handles=[Line2D([], [], color=INK, lw=1.2, label="unbiased")] +
                      [Line2D([], [], color=SEQ(0.3 + 0.65 * n / (len(vals) - 1)), label=fmt.format(v)) for n, v in enumerate(vals)],
                      loc="center left")
            continue
        cv = cvs[j]
        p = pmf[pmf.cv == cv]
        ax.plot(p.x, p.F_kT, color=INK, lw=1.2, zorder=3)
        for n, v in enumerate(vals):
            o = ov[(ov.family == fam) & (ov.cv == cv) & (ov.set_value == v)]
            ax.plot(o.x, o.F_plus_V_kT, color=SEQ(0.3 + 0.65 * n / (len(vals) - 1)), lw=1.0)
        ax.axvline(cvsum.loc[cv].barrier_pos, color=INK, lw=0.6, ls="--")
        ax.set_xlim(*XL[cv]); ax.set_ylim(0, 12)
        ax.set_xlabel(CVUNIT[cv])
        if j == 0:
            ax.set_ylabel(r"$F + \langle V\rangle$ ($k_{\rm B}T$)")
        fam_key = {"imetad": "imetad", "opes": "opesf", "ves": "vesf"}[fam]
        e = ovE[(ovE.family == fam_key) & (ovE.cv == cv)]
        if fam == "imetad":
            txt = r"$\varepsilon$: iMetaD, EATR" + "\n" + "\n".join(
                f"{int(pc)}: {num(e[(e.key == pc) & (e.estimator == 'iMetaD CDF')].err_ln.iloc[0], '+.1f')}, "
                f"{num(e[(e.key == pc) & (e.estimator == 'EATR CDF')].err_ln.iloc[0], '+.1f')}" for pc in vals)
        else:
            ee = e.set_index("estimator")
            txt = rf"$\gamma = 1$: {num(ee.loc['OPESf'].err_ln)}" + "\n" + rf"EATR-f: {num(ee.loc['EATRf'].err_ln)}, $\gamma$ = {ee.loc['EATRf'].gamma:.2f}"
        if fam == "imetad":
            ax.text(0.03 if cv == "Q" else 0.97, 0.03, txt, transform=ax.transAxes, ha="left" if cv == "Q" else "right", va="bottom")
        else:
            ax.text(0.97, 0.97, txt, transform=ax.transAxes, ha="right", va="top")
        label(ax, "abcdefghi"[k]); k += 1
save(fig, "figS_bias_overlays")

figS_bias_overlays: 6.50 x 5.80 in


## Figure S2. Growth law on the WT-MetaD grid

(a) Largest bias reached at a deposition point before escape, from the smallest recorded hill
height, against the growth law with the unbroadened hill overlap. (b) Coefficient of variation
of the escape times against the observed boost; dotted, the exponential value 1.

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(WSI, 2.3), layout="constrained")
ax = axes[0]
lim = (2, 150)
ax.plot(lim, lim, color=INK, lw=0.5)
for cv in CVS:
    g = dc[dc.cv == cv]
    ax.scatter(g.V_pred_unbroadened_kJmol, g.V_obs_kJmol, s=4, c=CVCOL[cv], alpha=0.7, lw=0, rasterized=True)
logx(ax, *lim); logy(ax, *lim)
ax.set_xlabel(r"predicted $V_n$ (kJ/mol)"); ax.set_ylabel(r"observed $V_{\rm obs}$ (kJ/mol)")
ax = axes[1]
for cv in CVS:
    g = dc[dc.cv == cv]
    ax.scatter(g.boost, g.cv_times, s=4, c=CVCOL[cv], alpha=0.7, lw=0, rasterized=True, label=CVNAME[cv])
ax.axhline(1, color=INK, lw=0.5, ls=":")
logx(ax, 1, 3e3); ax.set_ylim(0.2, 1.2)
ax.set_xlabel("observed boost"); ax.set_ylabel("escape-time spread / mean")
ax.legend(loc="upper right", handletextpad=0.1, markerscale=2)
for i, ax in enumerate(axes):
    label(ax, "ab"[i])
save(fig, "figS_growth_law")

figS_growth_law: 6.50 x 2.30 in


## Figure S3. OPES bias levels extended past the apparent barrier

Pace 1000, BARRIER 5 to 17 kJ/mol (apparent barrier along $Q$: 15.4 kJ/mol). (a) Observed rate
against acceleration; labels, BARRIER; lines, fits to 5–13 kJ/mol. (b) $\gamma = 1$ estimate of
each set (squares) and EATR-flooding over the sets up to that BARRIER (circles).

In [13]:
hs = pd.read_csv(DATA / "highbarrier_sets.csv")
hc = pd.read_csv(DATA / "highbarrier_conv.csv")
fig, axes = plt.subplots(1, 2, figsize=(WSI, 2.3), layout="constrained")
for cv in ("Q", "Ree"):
    d = hs[hs.cv == cv].sort_values("ln_exp_beta_v")
    ax = axes[0]
    ax.plot(d.ln_exp_beta_v, d.lnk_obs_us, "o", color=CVCOL[cv], label=rf"{CVNAME[cv]}, $\gamma$ = {d.gamma_all.iloc[0]:.2f}")
    lo = d[d.barrier <= 13]
    c = np.polyfit(lo.ln_exp_beta_v, lo.lnk_obs_us, 1)
    xx = np.linspace(d.ln_exp_beta_v.min() - 0.2, d.ln_exp_beta_v.max() + 0.2, 10)
    ax.plot(xx, np.polyval(c, xx), color=CVCOL[cv], lw=0.6)
    for _, r in d.iterrows():
        ax.annotate(str(int(r.barrier)), (r.ln_exp_beta_v, r.lnk_obs_us), xytext=(2, -8), textcoords="offset points")
    ax = axes[1]
    ax.plot(d.barrier, d.err_gamma1, "s--", color=CVCOL[cv], mfc="white", lw=0.7)
    h = hc[hc.cv == cv]
    ax.plot(h.max_barrier, h.err_eatrf, "o-", color=CVCOL[cv])
axes[0].set_xlabel(r"$\ln\langle e^{\beta V}\rangle$"); axes[0].set_ylabel(r"$\ln (k_{\rm obs}/\mu{\rm s}^{-1})$")
axes[0].legend(loc="upper left")
accuracy_band(axes[1])
axes[1].set_xlabel("BARRIER (kJ/mol)"); axes[1].set_ylabel(r"$\varepsilon$")
axes[1].legend(handles=[Line2D([], [], ls="--", marker="s", mfc="white", color=GREY, lw=0.7, label=r"$\gamma = 1$, per set"),
                        Line2D([], [], marker="o", color=GREY, label="EATR-f, sets up to BARRIER")], loc="lower left")
axes[1].set_ylim(-4.8, 1.3)
for i, ax in enumerate(axes):
    label(ax, "ab"[i])
save(fig, "figS_highbarrier_ladder")

figS_highbarrier_ladder: 6.50 x 2.30 in


## Figure S3b. OPES ladders at every pace, and the downturn past the barrier

One panel per CV: $\ln(k_{\rm obs}/k_0)$ against $\ln\langle e^{\beta V}\rangle$ for every BARRIER level at every pace (grey), with the pace-1000 ladder in colour, extended past the apparent barrier where those runs exist. Dashed diagonal: $\gamma = 1$, the naive estimate exact. Labels: BARRIER in kJ/mol.

In [14]:
osets = pd.read_csv(DATA / "opes_sets.csv")
hs = pd.read_csv(DATA / "highbarrier_sets.csv")
fig, axes = plt.subplots(1, 4, figsize=(WSI, 2.35), layout="constrained", sharex=True, sharey=True)
lim = (1.5, 8.6)
for j, cv in enumerate(CVS):
    ax = axes[j]
    ax.plot(lim, lim, color=INK, lw=0.6, ls="--")
    o = osets[osets.cv == cv]
    ax.plot(o.ln_exp_beta_v, o.lnk_obs_us - ref.ln_k, "o", color=GREY, mfc="none", ms=2.5, mew=0.6, ls="")
    d = hs[hs.cv == cv] if cv in set(hs.cv) else o[o.pace == 1000]
    d = d.sort_values("ln_exp_beta_v")
    ax.plot(d.ln_exp_beta_v, d.lnk_obs_us - ref.ln_k, "-", color=CVCOL[cv], lw=0.8)
    lo, hi = d[d.barrier <= 13], d[d.barrier > 13]  # levels past the barrier were run for this figure only
    ax.plot(lo.ln_exp_beta_v, lo.lnk_obs_us - ref.ln_k, "o", color=CVCOL[cv], ms=3.5, ls="")
    ax.plot(hi.ln_exp_beta_v, hi.lnk_obs_us - ref.ln_k, "o", color=CVCOL[cv], mfc="white", ms=3.5, ls="")
    for _, r in d.iterrows():
        ax.annotate(str(int(r.barrier)), (r.ln_exp_beta_v, r.lnk_obs_us - ref.ln_k), xytext=(3, -7),
                    textcoords="offset points", fontsize=7, color=CVCOL[cv])
    ax.set_title(CVNAME[cv])
    ax.set_xlabel(r"$\ln\langle e^{\beta V}\rangle$")
    label(ax, "abcd"[j])
axes[0].set_xlim(lim); axes[0].set_ylim(0.4, 6.2)
axes[0].set_ylabel(r"$\ln (k_{\rm obs}/k_0)$")
fig.legend(handles=[Line2D([], [], marker="o", color=GREY, mfc="none", ms=2.5, mew=0.6, ls="", label="every pace"),
                        Line2D([], [], marker="o", color=INK, ms=3.5, lw=0.8, label="pace 1000"),
                        Line2D([], [], marker="o", color=INK, mfc="white", ms=3.5, ls="", label="past the barrier"),
                        Line2D([], [], ls="--", lw=0.6, color=INK, label=r"$\gamma = 1$")], loc="outside lower center", ncol=4, columnspacing=1.5)
save(fig, "figS_opes_slopes")

figS_opes_slopes: 6.50 x 2.35 in


## Figure S3c. Does a failing Kolmogorov–Smirnov test flag the inaccurate estimates?

KS $p$ against deposition pace for every full-data fit: iMetaD (top) and EATR (middle) on the WT-MetaD grid, bias factors 2–20; naive OPES-flooding per level (bottom), coloured by BARRIER. Filled: within one ln unit of the reference; open: outside. Dashed: $p = 0.05$. Points are jittered in pace.

In [15]:
kg = pd.read_csv(DATA / "ks_grid.csv")
osets = pd.read_csv(DATA / "opes_sets.csv")
osets["err_ln"] = osets.lnk_obs_us - osets.ln_exp_beta_v - ref.ln_k
LEVELS = sorted(osets.barrier.unique())
PFLOOR = 1e-6
rng = np.random.default_rng(0)

def ks_points(ax, d, color):
    """Plot KS p against pace, filled within one ln unit and open outside, with a deterministic jitter in pace."""
    x = d.pace.values * np.exp(rng.uniform(-0.15, 0.15, len(d)))
    p = np.clip(d.ks_p.values, PFLOOR, 1.0)
    acc = (d.err_ln.abs() <= 1).values
    ax.plot(x[acc], p[acc], "o", color=color, ms=3, ls="")
    ax.plot(x[~acc], p[~acc], "o", mfc="none", mec=color, ms=3, mew=0.7, ls="")

fig, axes = plt.subplots(3, 4, figsize=(WSI, 4.9), layout="constrained", sharex=True, sharey=True)
for j, cv in enumerate(CVS):
    for i, (est, col) in enumerate((("iMetaD", G1), ("EATR", EATR))):
        ks_points(axes[i, j], kg[(kg.cv == cv) & (kg.estimator == est)], col)
    for k, lev in enumerate(LEVELS):
        ks_points(axes[2, j], osets[(osets.cv == cv) & (osets.barrier == lev)], SEQ(0.3 + 0.65 * k / (len(LEVELS) - 1)))
    axes[0, j].set_title(CVNAME[cv])
    axes[2, j].set_xlabel("pace (steps)")
for ax in axes.flat:
    ax.axhline(0.05, ls="--", lw=0.6, color=INK)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.yaxis.set_minor_formatter(NullFormatter())
axes[0, 0].set_ylim(PFLOOR / 3, 3); axes[0, 0].set_yticks([1e-6, 1e-4, 1e-2, 1])
for i, name in enumerate(("iMetaD", "EATR", "naive OPES")):
    axes[i, 0].set_ylabel(f"KS $p$, {name}")
    label(axes[i, 0], "abc"[i])
hs_ = [Line2D([], [], marker="o", color=INK, ms=3, ls="", label="within 1 ln unit"),
       Line2D([], [], marker="o", mfc="none", mec=INK, ms=3, mew=0.7, ls="", label="outside")]
hs_ += [Line2D([], [], marker="o", color=SEQ(0.3 + 0.65 * k / (len(LEVELS) - 1)), ms=3, ls="", label=f"BARRIER {int(lev)} kJ/mol")
        for k, lev in enumerate(LEVELS)]
fig.legend(handles=hs_, loc="outside lower center", ncol=4)
save(fig, "figS_ks_by_pace")

figS_ks_by_pace: 6.50 x 4.90 in


## Figure S3d. What the Kolmogorov–Smirnov test sees: the fitted CDFs

Empirical CDF of the 200 escape times (grey) and the fitted CDF (colour) for one mid-grid condition (bias factor 10, $h_0 = 0.25$ kJ/mol, $0.10\,k_{\rm B}T$) at four paces, every CV. Two figures: EATR against the raw escape time, whose fit depends on the raw time through each run's acceleration; iMetaD against the rescaled time $t\,\alpha$, whose fit is the exponential $1 - e^{-kt}$. Annotations: KS statistic $D$, KS $p$ and signed error $\varepsilon$.


In [16]:
cf = pd.read_csv(DATA / "cdf_fits.csv")
cm = pd.read_csv(DATA / "cdf_fits_meta.csv").set_index(["cv", "pace", "estimator"])
CDF_PACES = sorted(cf.pace.unique())

def ptext(p):
    """KS p as a short label: two decimals, or a power of ten below 0.01."""
    return f"$p$ = {p:.2f}" if p >= 0.01 else f"$p$ = $10^{{{int(np.floor(np.log10(p)))}}}$"

# one figure per estimator: empirical CDF (grey) and the fitted CDF (colour) against the
# estimator's own time variable, raw escape time for EATR and rescaled time for iMetaD
for est, col, xlab, name, sx in (("EATR", EATR, "escape time (ns)", "figS_cdf_fits", "col"),
                                 ("iMetaD", G1, r"rescaled time $t\,\alpha$ (ns)", "figS_cdf_fits_imetad", False)):
    # raw escape times span about a decade per pace, so the EATR columns share an axis; the
    # rescaled times span up to eight decades and differ by CV, so the iMetaD panels do not
    fig, axes = plt.subplots(4, 4, figsize=(WSI, 5.6), layout="constrained", sharey=True, sharex=sx)
    for i, cv in enumerate(CVS):
        for j, pace in enumerate(CDF_PACES):
            ax = axes[i, j]
            d = cf[(cf.cv == cv) & (cf.pace == pace) & (cf.estimator == est)].sort_values("time_ns")
            ax.plot(d.time_ns, d.ecdf, "o", color=GREY, ms=1.8, mew=0, ls="", zorder=1)
            ax.plot(d.time_ns, d.fit, "-", color=col, lw=1.0, zorder=2)
            m = cm.loc[(cv, pace, est)]
            # lower right is empty in every CDF panel
            ax.text(0.96, 0.05, f"$D$ = {m.ks_stat:.2f}\n{ptext(m.ks_p)}\n$\\varepsilon$ = {round(m.err_ln, 1) + 0.0:+.1f}",
                    transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color=col)
            ax.set_xscale("log"); ax.xaxis.set_minor_formatter(NullFormatter())
            if i == 0:
                ax.set_title(f"pace {pace:g}")
            if j == 0:
                ax.set_ylabel(f"{CVNAME[cv]}\nCDF")
            if i == 3:
                ax.set_xlabel(xlab)
            label(ax, "abcdefghijklmnop"[4 * i + j])
    axes[0, 0].set_ylim(0, 1.03); axes[0, 0].set_yticks([0, 0.5, 1])
    save(fig, name)


figS_cdf_fits: 6.50 x 5.60 in


figS_cdf_fits_imetad: 6.50 x 5.60 in


## Figure S4. Where EATR is less accurate than iMetaD

(a) Absolute error of both estimators for every full fit of the grid (200 runs); above the
diagonal EATR is worse. (b) Conditions with EATR worse, by pace and class: marginal (both within
one ln unit and differing by less than 0.3), statistical (reference inside the EATR interval or
difference below the subset spread), systematic (EATR outside one ln unit, iMetaD inside), other.

In [17]:
ev = pd.read_csv(DATA / "eatr_vs_imetad.csv")
wc = pd.read_csv(DATA / "eatr_worse_cases.csv")
fig, axes = plt.subplots(1, 2, figsize=(WSI, 2.3), layout="constrained")
ax = axes[0]
for cv in CVS:
    d = ev[ev.cv == cv]
    ax.scatter(d.err_imetad.abs(), d.err_eatr.abs(), s=4, c=CVCOL[cv], alpha=0.7, lw=0, rasterized=True, label=CVNAME[cv])
ax.plot([1e-3, 20], [1e-3, 20], color=INK, lw=0.5)
ax.axhline(1, color=INK, lw=0.5, ls=":"); ax.axvline(1, color=INK, lw=0.5, ls=":")
logx(ax, 1e-2, 12); logy(ax, 1e-2, 12)
ax.set_xlabel(r"iMetaD $|\varepsilon|$"); ax.set_ylabel(r"EATR $|\varepsilon|$")
ax.legend(loc="upper left", markerscale=2, handletextpad=0.1)
ax = axes[1]
order = [("marginal", "#BBBBBB"), ("statistical", "#56B4E9"), ("systematic", G1), ("other", INK)]
wc["short"] = wc["class"].str.split(" ").str[0]
ct = wc.groupby(["pace", "short"]).size().unstack(fill_value=0).reindex(columns=[o for o, _ in order], fill_value=0)
bottom = np.zeros(len(ct))
xl = [f"{int(p):.0e}".replace("e+0", "e") if p >= 1e4 else str(int(p)) for p in ct.index]
for c, col in order:
    ax.bar(xl, ct[c], bottom=bottom, color=col, label=c, width=0.7); bottom += ct[c].to_numpy()
ax.set_xlabel("pace (steps)"); ax.set_ylabel("conditions")
ax.set_ylim(0, bottom.max() * 1.45)
ax.tick_params(axis="x", rotation=45)
ax.legend(loc="upper left", ncol=2, columnspacing=0.8)
for i, ax in enumerate(axes):
    label(ax, "ab"[i])
save(fig, "figS_eatr_vs_imetad")

figS_eatr_vs_imetad: 6.50 x 2.30 in


## Figure S5. Error against the number of flooding sets retained

Sets sorted by acceleration; the $n$ lowest retained. Circles: EATR-flooding; squares:
$\gamma = 1$ (mean of the per-set estimates). Top: OPES, one curve per pace, 50 runs per set.
Bottom: VES, 200 runs per cutoff.

In [18]:
sc = pd.read_csv(DATA / "sets_scan.csv")
fig, axes = plt.subplots(2, 4, figsize=(WSI, 3.4), sharex="row", sharey=True, layout="constrained")
for j, cv in enumerate(CVS):
    for i, fam in enumerate(("opes", "ves")):
        ax = axes[i, j]
        g = sc[(sc.family == fam) & (sc.cv == cv)]
        for _, gg in (g.groupby("pace") if fam == "opes" else [(0, g)]):
            gg = gg.sort_values("n")
            ax.plot(gg.n, gg.err_eatrf, "-o", color=EATR, ms=2.5, lw=0.6, alpha=0.7)
            ax.plot(gg.n, gg.err_g1, "-s", color=G1, ms=2.5, lw=0.6, alpha=0.7)
        accuracy_band(ax)
        if i == 0:
            ax.set_title(CVNAME[cv])
        else:
            ax.set_xlabel("sets retained")
        label(ax, "abcdefgh"[4 * i + j])
axes[0, 0].set_ylabel(r"$\varepsilon$, OPES"); axes[1, 0].set_ylabel(r"$\varepsilon$, VES")
fig.legend(handles=[Line2D([], [], marker="o", color=EATR, ms=3, lw=0.6, label="EATR-flooding"),
                    Line2D([], [], marker="s", color=G1, ms=3, lw=0.6, label=r"$\gamma = 1$")], loc="outside lower center", ncol=2)
save(fig, "figS_sets_retained")

figS_sets_retained: 6.50 x 3.40 in


## Figure S6. VES cutoff flooding: validity checks

(a) Stage-1 bias at the walker for $Q$; dashed, the cutoff. (b) Maximum of the static stage-2
bias against the cutoff. (c) KS $p$ of the escape times against an exponential, per cutoff
($N = 200$). (d) Observed rate against acceleration; labels, cutoff. (e) Mean escape time per cutoff;
a non-monotonic sequence marks an unconverged stage 1.

In [19]:
s1 = pd.read_csv(DATA / "ves_stage1.csv")
vmax = pd.read_csv(DATA / "ves_static_max.csv")
vs = pd.read_csv(DATA / "ves_sets.csv")
fig, axes = plt.subplots(2, 3, figsize=(WSI, 3.8), layout="constrained")
ax = axes[0, 0]
for n, fc in enumerate((5, 9, 13, 15)):
    d = s1[s1.fc == fc]; col = SEQ(0.3 + 0.65 * n / 3)
    ax.plot(d.time_ns, d.bias_kJmol, color=col, lw=0.5, label=f"{fc}")
    ax.axhline(fc, color=col, lw=0.6, ls="--")
ax.set_xlabel("stage-1 time (ns)"); ax.set_ylabel("bias (kJ/mol)")
axes[1, 2].add_artist(axes[1, 2].legend(handles=ax.get_legend_handles_labels()[0], labels=[f"{fc} kJ/mol" for fc in (5, 9, 13, 15)],
                                        title=r"(a) cutoff $F_c$", title_fontsize=8, fontsize=8, handlelength=1.2,
                                        loc="center left", bbox_to_anchor=(-0.08, 0.5)))
ax = axes[0, 1]
ax.plot([4, 16], [4, 16], color=INK, lw=0.5)
for cv in CVS:
    d = vmax[vmax.cv == cv]
    ax.plot(d.fc, d.max_bias_kJmol, "o-", color=CVCOL[cv], label=CVNAME[cv])
ax.set_xlabel(r"$F_c$ (kJ/mol)"); ax.set_ylabel("max static bias (kJ/mol)"); ax.legend(loc="upper left")
ax = axes[0, 2]
for cv in CVS:
    d = vs[vs.cv == cv].sort_values("fc")
    ax.plot(d.fc, d.ks_p, "o-", color=CVCOL[cv])
ax.axhline(0.05, color=INK, lw=0.5, ls=":")
logy(ax, 1e-3, 1.5); ax.set_xlabel(r"$F_c$ (kJ/mol)"); ax.set_ylabel(r"KS $p$")
ax = axes[1, 0]
for cv in CVS:
    d = vs[vs.cv == cv].sort_values("ln_exp_beta_v")
    sl = np.polyfit(d.ln_exp_beta_v, d.lnk_obs_us, 1)[0]
    ax.plot(d.ln_exp_beta_v, d.lnk_obs_us, "o-", color=CVCOL[cv], label=f"{CVNAME[cv]}: {sl:.2f}")
ax.set_xlabel(r"$\ln\langle e^{\beta V}\rangle$"); ax.set_ylabel(r"$\ln (k_{\rm obs}/\mu{\rm s}^{-1})$")
axes[1, 2].legend(handles=ax.get_legend_handles_labels()[0], labels=ax.get_legend_handles_labels()[1],
                  title="(d) slope", title_fontsize=8, fontsize=8, handlelength=1.2, loc="center right", bbox_to_anchor=(1.0, 0.5))
ax = axes[1, 1]
for cv in CVS:
    d = vs[vs.cv == cv].sort_values("fc")
    ax.plot(d.fc, d.tau_obs_us * 1e3, "o-", color=CVCOL[cv])
logy(ax, 5, 1000); ax.set_xlabel(r"$F_c$ (kJ/mol)"); ax.set_ylabel("mean escape time (ns)")
axes[1, 2].axis("off")
for i, ax in enumerate(axes.ravel()[:5]):
    label(ax, "abcde"[i])
save(fig, "figS_ves_checks")

figS_ves_checks: 6.50 x 3.80 in


## Figure S7. EATR-flooding with censored runs

OPES bias levels at pace 1000 (five BARRIER values, 50 runs each), every run stopped at a common wall
time chosen so that 10, 25 or 50% of the slowest set is censored. Top: error with bootstrap 95%
intervals. Bottom: fitted $\gamma$.

In [20]:
cz = pd.read_csv(DATA / "censoring.csv")
fig, axes = plt.subplots(2, 2, figsize=(W1, 2.9), sharex="col", layout="constrained", gridspec_kw=dict(height_ratios=(1.5, 1)))
for j, cv in enumerate(("Q", "Ree")):
    d = cz[cz.cv == cv]
    x = d.censored_all.to_numpy() * 100
    ax = axes[0, j]
    accuracy_band(ax)
    for col, key, mk, lab in ((EATR, "eatrf", "o", "EATR-flooding"), (G1, "opesf", "s", r"$\gamma = 1$")):
        y = d[f"{key}_err"].to_numpy()
        ax.errorbar(x, y, yerr=np.vstack([y - d[f"{key}_lo"], d[f"{key}_hi"] - y]), color=col, marker=mk, ms=3.5, lw=1, capsize=2, elinewidth=0.6, label=lab)
    ax.set_title(CVNAME[cv]); ax.set_ylim(-3.2, 1.3)
    axes[1, j].errorbar(x, d.gamma, yerr=np.vstack([d.gamma - d.gamma_lo, d.gamma_hi - d.gamma]), color=EATR, marker="o", ms=3.5, lw=1, capsize=2, elinewidth=0.6)
    axes[1, j].set_ylim(0, 1.1)
    axes[1, j].set_xlabel("runs censored (%)")
axes[0, 0].set_ylabel(r"$\varepsilon$"); axes[1, 0].set_ylabel(r"$\gamma$")
axes[0, 0].legend(loc="lower left")
for i, ax in enumerate(axes.ravel()):
    label(ax, "abcd"[i])
save(fig, "figS_censoring")

figS_censoring: 3.37 x 2.90 in


## Figures S8-S11. The metadynamics grid, condition by condition

One figure per CV: signed error of both estimators on identical data, against the cost of
the condition, with one panel per bias factor. Colour is the deposition pace, shape the
hill height, open symbols iMetaD and filled symbols EATR.

In [21]:
from matplotlib.colors import LogNorm
from matplotlib.ticker import FixedLocator

PACES = [100, 500, 1000, 5000, 10000, 30000, 50000, 100000]
HEIGHTS = [0.1, 0.25, 0.5, 1.0]
MARK = {0.1: "o", 0.25: "s", 0.5: "^", 1.0: "D"}
PNORM = LogNorm(80, 1.3e5)
PCOL = {q: plt.cm.viridis(PNORM(q)) for q in PACES}
acg = pd.read_csv(DATA / "accuracy_cost.csv")
# untempered metadynamics is the b -> inf limit of the grid, so it joins it as a panel
acg = acg[acg.family.isin(["wtmetad", "untempered"])]

for cv in CVS:
    d = acg[acg.cv == cv]
    bfs = sorted(d.bf.unique())
    fig, axes = plt.subplots(1, len(bfs), figsize=(WSI, 2.3), sharey=True, layout="constrained")
    for ax, bf in zip(axes, bfs):
        g = d[d.bf == bf]
        accuracy_band(ax)
        for est, fill in (("iMetaD", False), ("EATR", True)):
            for h in HEIGHTS:
                for q in PACES:
                    r = g[(g.estimator == est) & (g.height == h) & (g.pace == q)]
                    if r.empty:
                        continue
                    ax.scatter(r.cost_us, r.err_ln, marker=MARK[h], s=16, zorder=2,
                               facecolors=PCOL[q] if fill else "none", edgecolors=PCOL[q], lw=0.7)
        ax.set_title(r"$b = \infty$" if not np.isfinite(bf) else rf"$b$ = {int(bf)}")
        logx(ax, 0.05, 150)
        ax.set_xlabel(r"cost ($\mu$s)")
    lo, hi = d.err_ln.min(), d.err_ln.max()
    pad = 0.06 * (hi - lo)
    axes[0].set_ylim(min(lo - pad, -1.6), max(hi + pad, 1.6))
    axes[0].set_ylabel(r"$\varepsilon = \ln k_{\rm est} - \ln k_0$")
    handles = [Line2D([], [], ls="", marker="o", ms=4, mfc="none", mec=INK, mew=0.7, label="iMetaD (naive)"),
               Line2D([], [], ls="", marker="o", ms=4, color=INK, label="EATR")]
    handles += [Line2D([], [], ls="", marker=MARK[h], ms=4, color=INK, label=rf"$h_0$ = {h}") for h in HEIGHTS]
    fig.legend(handles=handles, loc="outside lower center", ncol=3, columnspacing=1.0)
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=PNORM, cmap=plt.cm.viridis), ax=axes,
                      location="right", fraction=0.03, pad=0.01, aspect=18)
    cb.set_label(r"deposition pace $\tau$ (steps)")
    cb.set_ticks(FixedLocator(PACES))
    cb.ax.set_yticklabels([f"$10^{{{int(np.log10(q))}}}$" if np.log10(q) % 1 == 0 else "" for q in PACES])
    cb.ax.minorticks_off()
    cb.outline.set_linewidth(0.6)
    save(fig, f"figS_grid_{cv}")

figS_grid_Q: 6.50 x 2.30 in


figS_grid_RMSD: 6.50 x 2.30 in


figS_grid_Rg: 6.50 x 2.30 in


figS_grid_Ree: 6.50 x 2.30 in


## Figure S12. Frequency-adaptive deposition against a fixed pace


In [22]:
# The adaptive runs start at pace 100 and saturate at 1e5, and the grid holds both of those
# paces at the same bias factor and height, so the fixed-pace series between them is the
# baseline and the threshold alone decides where an adaptive run lands on it.
acf = pd.read_csv(DATA / "accuracy_cost.csv")
fix = acf[(acf.family == "wtmetad") & (acf.bf == 10) & (acf.height == 0.5)]
ada = acf[acf.family == "fametad"]
cvs = [c for c in CVS if c in set(ada.cv)]

fig, axes = plt.subplots(1, len(cvs), figsize=(WSI, 2.3), sharey=True, layout="constrained")
axes = np.atleast_1d(axes)
for ax, cv in zip(axes, cvs):
    accuracy_band(ax)
    for est, col in (("iMetaD", G1), ("EATR", EATR)):
        b = fix[(fix.cv == cv) & (fix.estimator == est)].sort_values("cost_us")
        ax.plot(b.cost_us, b.err_ln, "-o", color=col, lw=1.0, ms=3, mfc="white", mew=0.9,
                alpha=0.75, zorder=2)
        a = ada[(ada.cv == cv) & (ada.estimator == est)].sort_values("cost_us")
        ax.plot(a.cost_us, a.err_ln, "-^", color=col, lw=1.1, ms=5, zorder=4)
    ax.set_title(CVNAME[cv])
    ax.set_xlabel(COST)
    logx(ax, 0.1, 40)
axes[0].set_ylabel(r"$\varepsilon = \ln k_{\rm est} - \ln k_0$")
for i, ax in enumerate(axes):
    label(ax, "abcd"[i])
handles = [Line2D([], [], marker="o", ms=3, mfc="white", mew=0.9, color=INK, lw=1.0,
                  label="fixed pace"),
           Line2D([], [], marker="^", ms=5, color=INK, lw=1.1, label="frequency adaptive"),
           Line2D([], [], ls="", marker="s", ms=5, color=G1, label="iMetaD"),
           Line2D([], [], ls="", marker="s", ms=5, color=EATR, label="EATR")]
fig.legend(handles=handles, loc="outside lower center", ncol=4, columnspacing=1.2,
           handletextpad=0.4)
save(fig, "figS_fametad")


figS_fametad: 6.50 x 2.30 in
